# Composite MNIST detector — Google Colab

This notebook clones the project, downloads MNIST, generates the composite detection dataset, trains the 20-slot detector for 50 epochs, and visualizes a test prediction.

Before running it, replace `REPOSITORY_URL` with the URL of your pushed GitHub repository. In Colab, select **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# Confirm that Colab assigned a GPU.
!nvidia-smi

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# Replace this placeholder after pushing the repository to GitHub.
REPOSITORY_URL = 'https://github.com/your-username/your-repository.git'
PROJECT_DIR = '/content/mnist-detector'

!git clone $REPOSITORY_URL $PROJECT_DIR
%cd $PROJECT_DIR
!git log -1 --oneline

In [ ]:
# Torch and torchvision are preinstalled in Colab. Install the small optional utilities.
!pip -q install tqdm matplotlib

# Verify that the repository code is visible.
!find src src_model -maxdepth 2 -type f | sort

In [ ]:
# Download the original MNIST train/test data into data/MNIST/.
!python src/download_mnist.py.py

In [ ]:
# Generate 50,000 train, 10,000 validation, and 10,000 test examples.
# It creates data/uni_with_bboxes/{train,val,test}.pt.
!python src/create_multilabel_mnist_with_boxes.py --seed 42
!sed -n '1,220p' data/uni_with_bboxes/train_structure.md

In [ ]:
# Train the default compact model: 50 epochs, batch size 64, AdamW.
# The script automatically logs model parameter count, saves best/last checkpoints,
# writes history.csv and training_curves.png, and evaluates the test split.
!python -m src_model.train \
    --data-dir data/uni_with_bboxes \
    --model-size small \
    --epochs 50 \
    --batch-size 64 \
    --device auto

In [ ]:
# Show the loss and validation metrics saved by training.
from IPython.display import Image, display

display(Image(filename='outputs/mnist_detector_small/training_curves.png'))
!tail -n 6 outputs/mnist_detector_small/history.csv

In [ ]:
# Run confidence filtering plus class-aware NMS on a random test image.
!python -m src_model.predict \
    --checkpoint outputs/mnist_detector_small/best.pt \
    --data-dir data/uni_with_bboxes \
    --split test \
    --random \
    --output outputs/mnist_detector_small/random_test_prediction.png

display(Image(filename='outputs/mnist_detector_small/random_test_prediction.png'))

## Optional: resume after a disconnected runtime

Colab local files are removed after a runtime reset. To resume later, first copy `outputs/mnist_detector_small/` to Google Drive, then restore it and run:

```bash
python -m src_model.train --resume outputs/mnist_detector_small/last.pt --epochs 50
```